In [2]:
## 数据导入
import pandas as pd
import numpy as np

train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train.csv')  # 读取csv文件

In [3]:
train.info()  # 查看数据基本信息

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127744 entries, 0 to 127743
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   index            127744 non-null  int64  
 1   target           127744 non-null  int64  
 2   timestamp        127744 non-null  float64
 3   processId        127744 non-null  int64  
 4   threadId         127744 non-null  int64  
 5   parentProcessId  127744 non-null  int64  
 6   userId           127744 non-null  int64  
 7   mountNamespace   127744 non-null  int64  
 8   processName      127744 non-null  object 
 9   hostName         127744 non-null  int64  
 10  eventId          127744 non-null  int64  
 11  eventName        127744 non-null  object 
 12  stackAddresses   127744 non-null  object 
 13  argsNum          127744 non-null  int64  
 14  returnValue      127744 non-null  int64  
 15  args             127744 non-null  object 
dtypes: float64(1), int64(11), object(4)
me

In [4]:
## 特征工程（涉及 缺失数据， 数据编码， 异常数据处理etc）

In [5]:
### missing data imputation 
### using feature-engine
from sklearn.impute import SimpleImputer

# 查看缺失值
train.isnull().sum()

index              0
target             0
timestamp          0
processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
processName        0
hostName           0
eventId            0
eventName          0
stackAddresses     0
argsNum            0
returnValue        0
args               0
dtype: int64

In [6]:
train['args'][0] ## Python 的 list of dicts（字符串形式）

"[{'name': 'domain', 'type': 'int', 'value': 'AF_UNIX'}, {'name': 'type', 'type': 'int', 'value': 'SOCK_DGRAM|SOCK_CLOEXEC'}, {'name': 'protocol', 'type': 'int', 'value': 0}]"

In [7]:
import ast
import pandas as pd

# 假设 train 已经有 args 列
train['args_parsed'] = train['args'].apply(ast.literal_eval)


In [8]:
# 转换成 {name: value} 的字典
def list_to_dict(lst):
    return {d['name']: d['value'] for d in lst}

train['args_dict'] = train['args_parsed'].apply(list_to_dict)

# 展开成 DataFrame
args_df = pd.json_normalize(train['args_dict'])


In [9]:
args_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127744 entries, 0 to 127743
Data columns (total 38 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   domain      451 non-null    object 
 1   type        451 non-null    object 
 2   protocol    451 non-null    float64
 3   pathname    65560 non-null  object 
 4   flags       51280 non-null  object 
 5   dev         14324 non-null  float64
 6   inode       14324 non-null  float64
 7   fd          57149 non-null  float64
 8   statbuf     26285 non-null  object 
 9   dirfd       36772 non-null  float64
 10  mode        43398 non-null  object 
 11  ruid        4 non-null      float64
 12  euid        4 non-null      float64
 13  rgid        11 non-null     float64
 14  egid        11 non-null     float64
 15  cap         2832 non-null   object 
 16  sockfd      591 non-null    float64
 17  addr        591 non-null    object 
 18  addrlen     591 non-null    object 
 19  dirp        785 non-nul

In [10]:
train = pd.concat([train.drop(columns=['args', 'args_parsed', 'args_dict']), args_df], axis=1)


In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127744 entries, 0 to 127743
Data columns (total 53 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   index            127744 non-null  int64  
 1   target           127744 non-null  int64  
 2   timestamp        127744 non-null  float64
 3   processId        127744 non-null  int64  
 4   threadId         127744 non-null  int64  
 5   parentProcessId  127744 non-null  int64  
 6   userId           127744 non-null  int64  
 7   mountNamespace   127744 non-null  int64  
 8   processName      127744 non-null  object 
 9   hostName         127744 non-null  int64  
 10  eventId          127744 non-null  int64  
 11  eventName        127744 non-null  object 
 12  stackAddresses   127744 non-null  object 
 13  argsNum          127744 non-null  int64  
 14  returnValue      127744 non-null  int64  
 15  domain           451 non-null     object 
 16  type             451 non-null     obje

In [12]:
train.isnull().sum()

index                   0
target                  0
timestamp               0
processId               0
threadId                0
parentProcessId         0
userId                  0
mountNamespace          0
processName             0
hostName                0
eventId                 0
eventName               0
stackAddresses          0
argsNum                 0
returnValue             0
domain             127293
type               127293
protocol           127293
pathname            62184
flags               76464
dev                113420
inode              113420
fd                  70595
statbuf            101459
dirfd               90972
mode                84346
ruid               127740
euid               127740
rgid               127733
egid               127733
cap                124912
sockfd             127153
addr               127153
addrlen            127153
dirp               126959
count              126959
stack              127552
parent_tid         127552
child_tid   

In [13]:
train.size

6770432

In [14]:
# 5. 导出成新的 CSV
train.to_csv("D:/NUSMaster/semester1/CS5344BigDataAnalytic/Projects/sample_processes_train_parsed.csv", index=False)

print("✅ 已经生成新的 CSV 文件: sample_processes_train_parsed.csv")

✅ 已经生成新的 CSV 文件: sample_processes_train_parsed.csv


In [15]:
# 查看当前train中的数值型特征是否有缺失值
# 只选择数值型列
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
# 统计每个数值型列的缺失值个数
missing_numeric = train[num_cols].isnull().sum()
# 只显示有缺失的列
missing_numeric = missing_numeric[missing_numeric > 0]
print(missing_numeric)

target      127743
protocol    127293
dev         113420
inode       113420
fd           70595
dirfd        90972
ruid        127740
euid        127740
rgid        127733
egid        127733
sockfd      127153
count       126959
tls         127552
arg2        127333
arg3        127333
arg4        127333
arg5        127333
pid         127552
sig         127552
oldfd       127601
newfd       127612
uid         127737
gid         127741
dtype: int64


In [17]:
## 读取 parsed csv
train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train_parsed.csv')  # 读取csv文件
## 查看非数值型特征都有哪些列，每一列都有什么
cat_cols = train.select_dtypes(include=['object']).columns
for col in cat_cols:
    unique_values = train[col].unique()
    print(f"Column: {col}, Unique Values: {unique_values}\n")
    

Column: processName, Unique Values: ['systemd-resolve' 'systemd' 'systemd-network' 'sshd' 'systemd-journal'
 'dbus-daemon' 'systemd-logind' 'systemd-udevd' 'docker' 'dockerd' 'cron'
 '(sd-pam)' '(time-dir)' 'systemd-user-ru' 'systemd-timesyn'
 'containerd-shim' 'amazon-ssm-agen' 'ps' 'snapd' 'journal-offline'
 'kworker/dying' 'ssm-agent-worke' 'packagekitd' 'gmain' 'gdbus']

Column: eventName, Unique Values: ['socket' 'security_file_open' 'fstat' 'openat' 'close' 'setreuid'
 'setregid' 'cap_capable' 'accept4' 'getsockname' 'access'
 'sched_process_exit' 'bind' 'lstat' 'getdents64' 'security_inode_unlink'
 'unlink' 'connect' 'stat' 'clone' 'fchmod' 'prctl' 'kill' 'unlinkat'
 'umount' 'dup3' 'security_bprm_check' 'dup2' 'dup' 'setuid' 'execve'
 'setgid' 'accept']

Column: stackAddresses, Unique Values: ['[139913106282763, 139913103116537, 94901962555136]'
 '[140074839310116, 8103505641674583864]' '[140074839307913]' ...
 '[139903735520939, 8319683848551214897]'
 '[139903735523719, 2048, 

C:\Users\fkbgr\AppData\Local\Temp\ipykernel_34372\1138002083.py:2: DtypeWarning: Columns (47,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(r'D:\NUSMaster\semester1\CS5344BigDataAnalytic\Projects\sample_processes_train_parsed.csv')  # 读取csv文件
